In [1]:
import pandas as pd

df = pd.read_csv(r'Joins datasets/dataset_temporal_final.csv')

df.isna().sum()

Unnamed: 0               0
date                     0
home_team                0
away_team                0
home_score               0
away_score               0
tournament               0
city                     0
country                  0
neutral                  0
winner                   0
first_shooter            0
home_form_pts            0
away_form_pts            0
home_venue_type          0
home_aproveitamento      0
away_venue_type          0
away_aproveitamento      0
rest_days_diff           0
home_h2h_pts             0
away_h2h_pts             0
home_form_gd             0
away_form_gd             0
home_pen_win_rate        0
away_pen_win_rate        0
tournament_weight        0
home_Rank              123
home_TotalPoints         0
away_Rank              135
away_TotalPoints         0
home_GER                 0
home_ATQ                 0
home_MED                 0
home_DEF                 0
away_GER                 0
away_ATQ                 0
away_MED                 0
a

In [2]:
df.dropna(inplace=True)

In [3]:
df.drop(columns=['Unnamed: 0'], inplace=True)
df = pd.get_dummies(df, columns=['home_venue_type', 'away_venue_type'])
df.columns

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral', 'winner', 'first_shooter',
       'home_form_pts', 'away_form_pts', 'home_aproveitamento',
       'away_aproveitamento', 'rest_days_diff', 'home_h2h_pts', 'away_h2h_pts',
       'home_form_gd', 'away_form_gd', 'home_pen_win_rate',
       'away_pen_win_rate', 'tournament_weight', 'home_Rank',
       'home_TotalPoints', 'away_Rank', 'away_TotalPoints', 'home_GER',
       'home_ATQ', 'home_MED', 'home_DEF', 'away_GER', 'away_ATQ', 'away_MED',
       'away_DEF', 'home_venue_type_Casa', 'home_venue_type_Neutro',
       'away_venue_type_Fora', 'away_venue_type_Neutro'],
      dtype='str')

In [4]:
group_fixtures = pd.read_csv(r'group_fixtures.csv')
knockout_slots = pd.read_csv(r'knockout_slots.csv')
# Converter data para datetime
group_fixtures['date'] = pd.to_datetime(group_fixtures['date_utc'])

In [7]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import poisson
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SEÇÃO 1: CONFIGURAÇÃO GLOBAL E REPESCAGEM
# ============================================================
print("=" * 62)
print("SEÇÃO 1: CONFIGURAÇÃO GLOBAL E REPESCAGEM")
print("=" * 62)

MAX_GOLS = 8
N_SIMULACOES = 1000

playoff_map = {
    'FIFA Playoff 1': 'Congo DR',
    'FIFA Playoff 2': 'Iraq',
    'UEFA Playoff A': 'Bosnia and Herzegovina',
    'UEFA Playoff B': 'Sweden',
    'UEFA Playoff C': 'Turkey',
    'UEFA Playoff D': 'Czech Republic',
    'South Korea': 'Korea Republic',
    'United States': 'USA'
}

group_fixtures['home_team'] = group_fixtures['home_team'].replace(playoff_map)
group_fixtures['away_team'] = group_fixtures['away_team'].replace(playoff_map)

# ============================================================
# SEÇÃO 2: PREPARAÇÃO COM DADOS TÁTICOS
# ============================================================
print("Modelando dados para o LightGBM...")

colunas_necessarias = [
    'home_score', 'away_score',
    'home_GER', 'away_GER',
    'home_ATQ', 'home_MED', 'home_DEF',
    'away_ATQ', 'away_MED', 'away_DEF'
]
df_limpo = df.dropna(subset=colunas_necessarias).copy()
df_limpo['date'] = pd.to_datetime(df_limpo['date'])

cols_home = ['date', 'home_team', 'away_team', 'home_score', 'home_GER', 'away_GER',
             'home_form_pts', 'home_ATQ', 'home_MED', 'home_DEF',
             'away_ATQ', 'away_MED', 'away_DEF']
cols_unificadas = ['date', 'team', 'opponent', 'goals', 'team_GER', 'opponent_GER',
                   'team_form', 'team_ATQ', 'team_MED', 'team_DEF',
                   'opponent_ATQ', 'opponent_MED', 'opponent_DEF']

df_home = df_limpo[cols_home].copy()
df_home.columns = cols_unificadas
df_home['is_home'] = 1

cols_away = ['date', 'away_team', 'home_team', 'away_score', 'away_GER', 'home_GER',
             'away_form_pts', 'away_ATQ', 'away_MED', 'away_DEF',
             'home_ATQ', 'home_MED', 'home_DEF']
df_away = df_limpo[cols_away].copy()
df_away.columns = cols_unificadas
df_away['is_home'] = 0

poisson_data = pd.concat([df_home, df_away]).sort_values('date').reset_index(drop=True)

latest_stats = {}
for team in poisson_data['team'].unique():
    last_match = poisson_data[poisson_data['team'] == team].iloc[-1]
    latest_stats[team] = {
        'GER': last_match['team_GER'], 'form': last_match['team_form'],
        'ATQ': last_match['team_ATQ'], 'MED': last_match['team_MED'],
        'DEF': last_match['team_DEF']
    }

default_stats = {
    'GER': poisson_data['team_GER'].mean(), 'form': 5.0,
    'ATQ': poisson_data['team_ATQ'].mean(), 'MED': poisson_data['team_MED'].mean(),
    'DEF': poisson_data['team_DEF'].mean()
}

# ============================================================
# SEÇÃO 3: O MOTOR LIGHTGBM (CORREÇÃO APLICADA)
# ============================================================
print("Treinando Inteligência Artificial LightGBM...")

features = [
    'is_home', 'team_GER', 'opponent_GER', 'team_form',
    'team_ATQ', 'team_MED', 'team_DEF',
    'opponent_ATQ', 'opponent_MED', 'opponent_DEF'
]

X = poisson_data[features].astype('float32')
y = poisson_data['goals'].astype('float32')

# Peso temporal: Jogos antigos valem menos que jogos recentes
dias_atras = (poisson_data['date'].max() - poisson_data['date']).dt.days
pesos = np.exp(-dias_atras / 1095).astype('float32')

# INSTANCIAR PRIMEIRO, TREINAR DEPOIS (Bug corrigido)
modelo_lgbm = lgb.LGBMRegressor(
    objective='poisson',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
    n_jobs=1,
    force_col_wise=True
)

modelo_lgbm.fit(X, y, sample_weight=pesos)
print("✅ LightGBM treinado com sucesso (Pesos Temporais Aplicados)!\n")

def get_lambda_lgbm(team_attack, team_defense, is_home):
    t = latest_stats.get(team_attack, default_stats)
    d = latest_stats.get(team_defense, default_stats)
    X_pred = pd.DataFrame([{
        'is_home': is_home,
        'team_GER': t['GER'], 'opponent_GER': d['GER'], 'team_form': t['form'],
        'team_ATQ': t['ATQ'], 'team_MED': t['MED'], 'team_DEF': t['DEF'],
        'opponent_ATQ': d['ATQ'], 'opponent_MED': d['MED'], 'opponent_DEF': d['DEF']
    }])
    return max(0.1, modelo_lgbm.predict(X_pred)[0])

def simular_partida(home_team, away_team, neutral=True):
    lam = get_lambda_lgbm(home_team, away_team, is_home=0 if neutral else 1)
    mu  = get_lambda_lgbm(away_team, home_team, is_home=0)

    h = int(min(np.random.poisson(lam), MAX_GOLS))
    a = int(min(np.random.poisson(mu),  MAX_GOLS))

    return h, a

# ============================================================
# SEÇÃO 4: MOTOR MONTE CARLO (MÓDULO DE GRUPOS)
# ============================================================
print("=" * 62)
print(f"🎲 MONTE CARLO — {N_SIMULACOES} SIMULAÇÕES (FASE DE GRUPOS)")
print("=" * 62)

grupos = group_fixtures.groupby('group').apply(lambda g: list(set(g['home_team'].tolist() + g['away_team'].tolist()))).to_dict()
jogos_grupo = group_fixtures[['group', 'home_team', 'away_team']].to_dict('records')

np.random.seed(42)

acc_placar_grupo = defaultdict(Counter)
acc_posicao      = defaultdict(Counter)
acc_avanco       = Counter()

for n in range(N_SIMULACOES):
    if (n + 1) % 200 == 0:
        print(f"  ▶ {n + 1}/{N_SIMULACOES} simulações concluídas...")

    terceiros_deste_universo = []

    for grupo, times in grupos.items():
        tabela = {t: {'Pts': 0, 'GF': 0, 'GC': 0, 'SG': 0} for t in times}

        for jogo in [j for j in jogos_grupo if j['group'] == grupo]:
            ht, at = jogo['home_team'], jogo['away_team']
            h, a = simular_partida(ht, at, neutral=True)

            acc_placar_grupo[(ht, at)][(h, a)] += 1

            tabela[ht]['GF'] += h; tabela[ht]['GC'] += a; tabela[ht]['SG'] += h - a
            tabela[at]['GF'] += a; tabela[at]['GC'] += h; tabela[at]['SG'] += a - h

            if   h > a:  tabela[ht]['Pts'] += 3
            elif h == a: tabela[ht]['Pts'] += 1; tabela[at]['Pts'] += 1
            else:        tabela[at]['Pts'] += 3

        classif = sorted(tabela.items(), key=lambda x: (x[1]['Pts'], x[1]['SG'], x[1]['GF']), reverse=True)

        for pos, (team, _) in enumerate(classif, 1):
            acc_posicao[(grupo, team)][pos] += 1

        acc_avanco[classif[0][0]] += 1
        acc_avanco[classif[1][0]] += 1
        terceiros_deste_universo.append((classif[2][0], classif[2][1], grupo))

    melhores_3os = sorted(terceiros_deste_universo, key=lambda x: (x[1]['Pts'], x[1]['SG'], x[1]['GF']), reverse=True)[:8]
    for time_3o, _, _ in melhores_3os:
        acc_avanco[time_3o] += 1

print(f"\n✅ {N_SIMULACOES} simulações da Fase de Grupos concluídas!\n")

# ============================================================
# SEÇÃO 5: FASE DE GRUPOS — EXIBIÇÃO E EXTRAÇÃO DA MODA
# ============================================================
print("=" * 62)
print("📊 FASE DE GRUPOS — PROBABILIDADES DE CLASSIFICAÇÃO")
print("=" * 62)

def _pos_media(grupo, team):
    c   = acc_posicao.get((grupo, team), Counter({1: 1}))
    tot = sum(c.values())
    return sum(p * v for p, v in c.items()) / tot if tot > 0 else 4.0

# Inicialização das variáveis estruturais da "Moda" para o Mata-Mata
modo_primeiros = {}
modo_segundos = {}
modo_terceiros_lista = []

for grupo in sorted(grupos.keys()):
    print(f"\n{'─' * 62}")
    print(f"  GRUPO {grupo}")
    print(f"{'─' * 62}")

    times_g    = grupos[grupo]
    classif_mc = sorted(times_g, key=lambda t: _pos_media(grupo, t))

    # --- PERSISTÊNCIA DA CONFIGURAÇÃO MAIS FREQUENTE ---
    modo_primeiros[grupo] = classif_mc[0]
    modo_segundos[grupo]  = classif_mc[1]
    modo_terceiros_lista.append((classif_mc[2], None, grupo))

    # Tabela de Saída do Terminal
    print(f"  {'Pos':<4} {'Seleção':<20}  {'1º%':>4}  {'2º%':>4}  {'3º%':>4}  {'4º%':>4} | {'Avança%':>6}")
    print(f"  {'─' * 60}")
    for rank, team in enumerate(classif_mc, 1):
        c   = acc_posicao.get((grupo, team), Counter())
        tot = sum(c.values()) or 1
        p1  = c.get(1, 0) / tot * 100
        p2  = c.get(2, 0) / tot * 100
        p3  = c.get(3, 0) / tot * 100
        p4  = c.get(4, 0) / tot * 100
        p_avanca = (acc_avanco[team] / N_SIMULACOES) * 100

        icon = "✅" if p_avanca > 50 else "❌"
        print(f"  {icon} {rank:<2} {team:<20}  {p1:>3.0f}%  {p2:>3.0f}%  {p3:>3.0f}%  {p4:>3.0f}% | {p_avanca:>6.1f}%")

    print(f"\n  ┌─ PLACARES MAIS FREQUENTES ─────────────────────────┐")
    for jogo in [j for j in jogos_grupo if j['group'] == grupo]:
        ht, at = jogo['home_team'], jogo['away_team']
        cnt    = acc_placar_grupo.get((ht, at), Counter())
        if not cnt: continue
        total  = sum(cnt.values())
        top3   = cnt.most_common(3)
        top3_str = "  |  ".join(f"{h}-{a} ({v / total * 100:.0f}%)" for (h, a), v in top3)
        print(f"  │  {ht:20s} vs {at}")
        print(f"  │  Moda:  {top3_str}")
        print(f"  │")
    print(f"  └{'─' * 52}┘")

# --- RESOLUÇÃO INDEPENDENTE DOS 8 MELHORES TERCEIROS DA MODA ---
# Ordena os 12 terceiros colocados com base na taxa real de avanço obtida na simulação
modo_melhores_3os = sorted(
    modo_terceiros_lista,
    key=lambda x: acc_avanco.get(x[0], 0),
    reverse=True
)[:8]

print("\n" + "=" * 62)
print("💾 ARQUIVAMENTO DA FASE DE GRUPOS CONCLUÍDO COM SUCESSO!")
print("=" * 62)
print("Os estados determinísticos da simulação foram salvos nas variáveis globais.")
print(f"-> 3ºs Lugares Selecionados para Avançar: {[t[0] for t in modo_melhores_3os]}")

# ============================================================
# SEÇÃO 6: EXPORTAÇÃO KAGGLE (FASE DE GRUPOS)
# ============================================================
print("\n" + "=" * 62)
print("📁 GERANDO CSV DE SUBMISSÃO KAGGLE (Fase de Grupos)")
print("=" * 62)

previsoes_grupos = []

# Iteramos sobre a base oficial de jogos da fase de grupos
for index, row in group_fixtures.iterrows():
    match_id = row['match_id']
    ht = row['home_team']
    at = row['away_team']

    # Recupera a contagem de placares (os 1000 universos simulados para este jogo)
    cnt = acc_placar_grupo.get((ht, at), Counter())

    if cnt:
        # Pegamos o placar mais provável (a Moda Absoluta do Monte Carlo)
        (h_moda, a_moda), _ = cnt.most_common(1)[0]
    else:
        h_moda, a_moda = 0, 0 # Fallback de segurança

    # Regras de Formatação do Kaggle:
    # 1. Score: Formato exato "H-A"
    score_str = f"{h_moda}-{a_moda}"

    # 2. Winning team: "home", "away" ou "draw"
    if h_moda > a_moda:
        winner = 'home'
    elif a_moda > h_moda:
        winner = 'away'
    else:
        winner = 'draw'

    # Preenchemos a linha da tabela respeitando exatamente a ordem pedida
    previsoes_grupos.append({
        'match_id': match_id,
        'Score': score_str,
        'Winning team': winner
    })

# Criar o DataFrame final
df_kaggle_grupos = pd.DataFrame(previsoes_grupos)

# Salvar em CSV (sem a coluna de index, como o Kaggle exige)
nome_arquivo = "submission_fase_grupos.csv"
df_kaggle_grupos.to_csv(nome_arquivo, index=False)

print(f"✅ Arquivo '{nome_arquivo}' gerado com sucesso na sua pasta!")
display(df_kaggle_grupos.head(10))

SEÇÃO 1: CONFIGURAÇÃO GLOBAL E REPESCAGEM
Modelando dados para o LightGBM...
Treinando Inteligência Artificial LightGBM...
✅ LightGBM treinado com sucesso (Pesos Temporais Aplicados)!

🎲 MONTE CARLO — 1000 SIMULAÇÕES (FASE DE GRUPOS)
  ▶ 200/1000 simulações concluídas...
  ▶ 400/1000 simulações concluídas...
  ▶ 600/1000 simulações concluídas...
  ▶ 800/1000 simulações concluídas...
  ▶ 1000/1000 simulações concluídas...

✅ 1000 simulações da Fase de Grupos concluídas!

📊 FASE DE GRUPOS — PROBABILIDADES DE CLASSIFICAÇÃO

──────────────────────────────────────────────────────────────
  GRUPO A
──────────────────────────────────────────────────────────────
  Pos  Seleção                1º%   2º%   3º%   4º% | Avança%
  ────────────────────────────────────────────────────────────
  ✅ 1  Mexico                 31%   30%   25%   14% |   79.4%
  ✅ 2  Korea Republic         32%   26%   25%   17% |   77.0%
  ✅ 3  Czech Republic         31%   28%   23%   18% |   75.2%
  ❌ 4  South Africa       

,match_id,Score,Winning team
0,1,1-0,home
1,2,1-1,draw
2,3,0-0,draw
3,4,1-0,home
4,5,0-1,away
5,6,0-1,away
6,7,0-0,draw
7,8,0-2,away
8,9,2-0,home
9,10,1-0,home


In [10]:
# ============================================================
# SEÇÃO 9: MATA-MATA DETERMINÍSTICO E EXPORTAÇÃO KAGGLE
# ============================================================
import re
import numpy as np
import pandas as pd

print("=" * 62)
print("⚔️ SIMULAÇÃO DO MATA-MATA (Caminho da Moda dos Grupos)")
print("=" * 62)

# 1. Função Determinística (Calcula o placar matemático mais provável)
def simular_partida_deterministica(home_team, away_team):
    lam = get_lambda_lgbm(home_team, away_team, is_home=0) # is_home=0 pois a Copa é campo neutro
    mu  = get_lambda_lgbm(away_team, home_team, is_home=0)

    goals = np.arange(MAX_GOLS + 1)
    matrix = np.outer(poisson.pmf(goals, lam), poisson.pmf(goals, mu))
    idx = np.argmax(matrix)
    h, a = divmod(idx, MAX_GOLS + 1)
    return h, a

# 2. Estruturas de Rastreamento
w_bracket = {}
l_bracket = {}
alocados  = set()
previsoes_ko = []

# Lógica de alocação dos 3ºs lugares
def alocar_3o(slot_desc):
    match = re.search(r'\((.*?)\)', str(slot_desc))
    if match:
        grupos_permitidos = match.group(1).split('/')
        for t, _, grp in modo_melhores_3os:
            if grp in grupos_permitidos and t not in alocados:
                alocados.add(t); return t
    for t, _, grp in modo_melhores_3os:
        if t not in alocados:
            alocados.add(t); return t
    return 'TBD'

# Lógica de resolução do Slot
def resolver(slot_desc):
    s = str(slot_desc).strip()
    if s.startswith('Winner Group'):    return modo_primeiros.get(s[-1], 'TBD')
    if s.startswith('Runner-up Group'): return modo_segundos.get(s[-1], 'TBD')
    if s.startswith('Best 3rd'):        return alocar_3o(s)
    if s.startswith('Winner Match'):    return w_bracket.get(int(s.split()[-1]), 'TBD')
    if s.startswith('Loser Match'):     return l_bracket.get(int(s.split()[-1]), 'TBD')
    return 'TBD'

fase_atual = ""

# 3. Executando o Chaveamento
knockout_slots = knockout_slots.sort_values('match_id')

for _, row in knockout_slots.iterrows():
    m_id = row['match_id']
    fase = row['round']
    ht = resolver(row['slot_home'])
    at = resolver(row['slot_away'])

    if fase != fase_atual:
        print(f"\n{'═' * 62}")
        print(f"  {fase.upper()}")
        print(f"{'═' * 62}")
        fase_atual = fase

    if ht != 'TBD' and at != 'TBD':
        h, a = simular_partida_deterministica(ht, at)

        # Lógica de Pênaltis (Kaggle exige True/False)
        penalties = False
        if h > a:
            vencedor, perdedor = ht, at
            win_team_str = 'home'
        elif a > h:
            vencedor, perdedor = at, ht
            win_team_str = 'away'
        else:
            # Empate: Decide quem avança pela força tática (GER) e marca Pênaltis como True
            penalties = True
            ger_h = latest_stats.get(ht, default_stats)['GER']
            ger_a = latest_stats.get(at, default_stats)['GER']
            if ger_h >= ger_a:
                vencedor, perdedor = ht, at
                win_team_str = 'home'
            else:
                vencedor, perdedor = at, ht
                win_team_str = 'away'

        w_bracket[m_id] = vencedor
        l_bracket[m_id] = perdedor

        # Display Terminal
        resultado_str = f"  [{m_id:>3}] {ht:20s} {h} x {a}  {at}"
        print(resultado_str)
        if penalties:
            print(f"        ↳ {vencedor} vence nos pênaltis")

        # Salvar linha para o Kaggle
        previsoes_ko.append({
            'match_id': m_id,
            'home_team': ht,
            'away_team': at,
            'Score': f"{h}-{a}",
            'Winning team': win_team_str,
            'Penalties': penalties
        })

print("\n" + "🌟" * 31)
print(f"🏆 O GRANDE CAMPEÃO: {w_bracket[max(w_bracket.keys())].upper()} 🏆")
print("🌟" * 31)

# ============================================================
# 4. CONSTRUÇÃO DO CSV FINAL ÚNICO
# ============================================================
print("\n" + "=" * 62)
print("📁 GERANDO ARQUIVO DE SUBMISSÃO FINAL (GRUPOS + MATA-MATA)")
print("=" * 62)

# Convertendo o Mata-Mata para DataFrame
df_kaggle_ko = pd.DataFrame(previsoes_ko)

# Adicionando as colunas extras exigidas pelo formato Kaggle no df dos grupos
# (Assumindo que df_kaggle_grupos já está na memória da célula anterior)
df_kaggle_grupos_completo = df_kaggle_grupos.copy()

# O Kaggle precisa das colunas home_team e away_team para todas as linhas
df_kaggle_grupos_completo['home_team'] = group_fixtures['home_team']
df_kaggle_grupos_completo['away_team'] = group_fixtures['away_team']
# Na fase de grupos não há pênaltis
df_kaggle_grupos_completo['Penalties'] = False

# Reordenar as colunas para o padrão exato do Kaggle
colunas_ordem = ['match_id', 'home_team', 'away_team', 'Score',  'Winning team', 'Penalties']
df_kaggle_grupos_completo = df_kaggle_grupos_completo[colunas_ordem]
df_kaggle_ko = df_kaggle_ko[colunas_ordem]

# Unir Fase de Grupos + Mata-Mata
df_submissao_final = pd.concat([df_kaggle_grupos_completo, df_kaggle_ko], ignore_index=True)

# Salvar arquivo
nome_arquivo_final = "Kaggle_Submission_Complete.csv"
df_submissao_final.to_csv(nome_arquivo_final, index=False)

print(f"✅ Arquivo '{nome_arquivo_final}' criado com sucesso com as 104 partidas!")
display(df_submissao_final.tail(10)) # Mostra os últimos 10 jogos (Finais) para você ver o formato

⚔️ SIMULAÇÃO DO MATA-MATA (Caminho da Moda dos Grupos)

══════════════════════════════════════════════════════════════
  ROUND OF 32
══════════════════════════════════════════════════════════════
  [ 73] Korea Republic       1 x 1  Canada
        ↳ Korea Republic vence nos pênaltis
  [ 74] Morocco              1 x 0  Sweden
  [ 75] Germany              1 x 1  Scotland
        ↳ Germany vence nos pênaltis
  [ 76] Netherlands          1 x 0  Brazil
  [ 77] Côte d'Ivoire        1 x 1  Norway
        ↳ Norway vence nos pênaltis
  [ 78] France               1 x 0  Japan
  [ 79] Mexico               0 x 1  Senegal
  [ 80] England              1 x 0  Czech Republic
  [ 81] Belgium              1 x 0  Ecuador
  [ 82] Turkey               1 x 0  Paraguay
  [ 83] Colombia             0 x 1  Croatia
  [ 84] Spain                1 x 1  Austria
        ↳ Spain vence nos pênaltis
  [ 85] Switzerland          1 x 0  New Zealand
  [ 86] USA                  1 x 0  Egypt
  [ 87] Argentina            1 

,match_id,home_team,away_team,Score,Winning team,Penalties
94,95,USA,Portugal,0-1,away,False
95,96,Switzerland,Argentina,0-1,away,False
96,97,Germany,Morocco,0-0,home,True
97,98,Spain,Belgium,1-0,home,False
98,99,France,England,1-1,home,True
99,100,Portugal,Argentina,1-1,home,True
100,101,Germany,Spain,1-1,away,True
101,102,France,Portugal,1-1,home,True
102,103,Germany,Portugal,1-1,away,True
103,104,Spain,France,1-1,home,True
